# Notebook 1: Data Collection

## Overview

This notebook fetches annual GDP growth data for 10 MENA countries from the World Bank Open Data API and constructs the panel dataset used throughout the analysis.

**Indicator**: GDP growth, annual % — `NY.GDP.MKTP.KD.ZG`  
**Countries**: Algeria, Bahrain, Egypt, Jordan, Kuwait, Morocco, Oman, Qatar, Saudi Arabia, Tunisia  
**Period**: 1990–2023 (34 annual observations per country)

---

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import wbgapi as wb
import os
import warnings
warnings.filterwarnings('ignore')

from src.data_processing import (
    MENA_COUNTRIES, GDP_GROWTH_INDICATOR,
    START_YEAR, END_YEAR,
    fetch_gdp_growth, build_panel,
)
from src.visualization import plot_gdp_growth_panel

os.makedirs('../data/raw', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../results/figures', exist_ok=True)
os.makedirs('../results/tables', exist_ok=True)

## Countries and Indicator

The study uses 10 MENA economies with continuous GDP growth series from 1990 to 2023. The choice reflects data availability and economic diversity across oil-exporting and oil-importing countries in the region.


In [ ]:
print(f'Indicator: {GDP_GROWTH_INDICATOR}')
print(f'Period   : {START_YEAR}–{END_YEAR}')
print(f'Countries:')
for code, name in MENA_COUNTRIES.items():
    print(f'  {code}  {name}')

## Fetch Data from World Bank API

Data is retrieved programmatically via `wbgapi`. The raw response is reshaped into a long-format panel and saved before any transformation.


In [ ]:
panel_raw = fetch_gdp_growth()
print(f'Shape: {panel_raw.shape}')
print(f'Years: {panel_raw["year"].min()} – {panel_raw["year"].max()}')
print(f'Countries: {panel_raw["country_code"].nunique()}')
panel_raw.head(12)

## Missing Data Assessment

Isolated gaps (at most 2 consecutive years) are filled via linear interpolation within each country series. Series with more than 10% missing observations are flagged for review.


In [ ]:
missing_stats = panel_raw.groupby('country_code')['gdp_growth'].apply(
    lambda s: s.isnull().sum()
).reset_index()
missing_stats.columns = ['country_code', 'missing_count']
missing_stats['missing_pct'] = (missing_stats['missing_count'] / 34 * 100).round(1)
missing_stats['country_name'] = missing_stats['country_code'].map(MENA_COUNTRIES)
print(missing_stats[['country_code', 'country_name', 'missing_count', 'missing_pct']].to_string(index=False))

## Panel Construction

The raw panel is passed through `build_panel`, which applies within-country interpolation and returns a sorted, complete panel.


In [ ]:
panel = build_panel(panel_raw)
print(f'Panel shape after cleaning: {panel.shape}')
panel.to_csv('../data/raw/gdp_growth_raw.csv', index=False)
panel.to_csv('../data/processed/mena_gdp_panel.csv', index=False)
print('Saved to data/raw/gdp_growth_raw.csv and data/processed/mena_gdp_panel.csv')

## Panel Overview

Summary statistics reveal the distributional characteristics of GDP growth across the MENA panel. High standard deviations reflect the region's macroeconomic volatility.


In [ ]:
summary = panel.groupby('country_name')['gdp_growth'].describe().round(2)
print(summary)

## Temporal Trends

The panel plot below shows the GDP growth trajectory for each country from 1990 to 2023. Notable events — the 2008 financial crisis, 2014–2016 oil price collapse, and 2020 COVID shock — are visible as synchronised contractions across most countries.


In [ ]:
plot_gdp_growth_panel(
    panel,
    save_path='../results/figures/01_gdp_growth_panel.png',
)

## Summary

The panel covers 340 country-year observations (10 countries × 34 years) with GDP growth ranging from approximately −30% (Kuwait 1990, Gulf War) to +26% (Qatar 2022). Notebook 2 proceeds with stationarity testing and time series diagnostics.
